# ЛР 4. Генерация признаков и baseline-модель

**Датасет:** Marvel. Продолжение [ЛР 3](../LR3_marvel/LR3_marvel.ipynb). Одна строка — конфигурация объекта; предсказываем отопительную и охлаждающую нагрузки.

**Задачи:** получить дополнительные признаки для train и test, проверить масштабы признаков и обучить простую модель для отсчёта качества. Разбиение ЛР 1 сохраняется: 614 train и 154 test.

## 1. Загрузка подготовленных данных

Входы и цели берём из результатов ЛР 3. `source_row` — идентификатор исходной строки, а не вход модели. Все настройки преобразований выполняются на train.

In [1]:
from pathlib import Path
import platform
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

source = Path("../LR3_marvel/processed")
if not source.exists():
    source = Path("LR3_marvel/processed")
assert source.exists(), "Не найдены результаты ЛР 3"
def read(name):
    return pd.read_csv(source / f"{name}.csv", sep=";", index_col="source_row")
X_train, X_test = read("X_train"), read("X_test")
y_train, y_test = read("y_train"), read("y_test")
assert X_train.shape == (614, 16) and X_test.shape == (154, 16)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.columns.equals(X_test.columns)
assert not X_train.isna().any().any() and not X_test.isna().any().any()
print("Python:", platform.python_version())
for package in ["numpy", "pandas", "scikit-learn"]:
    print(package, version(package))
print("Входы:", X_train.shape, X_test.shape)
print("Цели:", list(y_train.columns))

Python: 3.12.5
numpy 2.4.5
pandas 3.0.2
scikit-learn 1.8.0
Входы: (614, 16) (154, 16)
Цели: ['Heating Load', 'Cooling Load']


# Раздел 2. Генерация новых признаков

## 2.1. Новые характеристики

Создаём три дополнительных признака по известным входам:

| Признак | Формула | Смысл |
|---|---|---|
| Wall Surface Ratio | площадь стен / площадь поверхности | доля стен в площади поверхности |
| Roof Surface Ratio | площадь крыши / площадь поверхности | доля крыши в площади поверхности |
| Compactness × Glazing | компактность × площадь остекления | совместное влияние формы и доли остекления |

Это исследовательские характеристики, а не проверенные физические законы. Используем только входные параметры, доступные для нового объекта. Одинаковые формулы применяем к train и test; целевые нагрузки в расчётах не участвуют.

In [2]:
def add_features(frame):
    result = frame.copy()
    assert (result["Surface Area"] > 0).all(), "Деление на ноль в площади поверхности"
    result["Wall Surface Ratio"] = result["Reactor Wall Area"] / result["Surface Area"]
    result["Roof Surface Ratio"] = result["Reactor Roof Area"] / result["Surface Area"]
    result["Compactness × Glazing"] = result["Relative Compactness"] * result["Glazing Area"]
    return result

X_train_generated = add_features(X_train)
X_test_generated = add_features(X_test)
new_features = ["Wall Surface Ratio", "Roof Surface Ratio", "Compactness × Glazing"]
assert X_train_generated.columns.equals(X_test_generated.columns)
assert np.isfinite(X_train_generated.to_numpy()).all()
assert np.isfinite(X_test_generated.to_numpy()).all()
display(X_train_generated[new_features].head())
display(X_train_generated[new_features].describe().T)
print("Размер до и после:", X_train.shape, X_train_generated.shape, X_test_generated.shape)

,Wall Surface Ratio,Roof Surface Ratio,Compactness × Glazing
source_row,,,
60,0.520000,0.240000,0.082
618,0.437500,0.281250,0.256
346,0.500000,0.250000,0.215
294,0.565217,0.217391,0.225
231,0.419355,0.290323,0.066


,count,mean,std,min,25%,50%,75%,max
Wall Surface Ratio,614.0,0.479934,0.081499,0.357143,0.404839,0.454545,0.538462,0.629630
Roof Surface Ratio,614.0,0.260033,0.040750,0.185185,0.230769,0.272727,0.297581,0.321429
Compactness × Glazing,614.0,0.179456,0.106181,0.000000,0.076750,0.177500,0.264000,0.392000


Размер до и после: (614, 16) (614, 19) (154, 19)


## 2.2. Проверка масштаба и стандартизация

Площади измеряются сотнями, а относительные величины и бинарные признаки — около единицы. Такая разница может влиять на методы, использующие расстояния или регуляризацию. Сравним диапазоны числовых столбцов на train.

Применяем `StandardScaler` к шести исходным числовым и трём новым признакам: параметры вычисляются только на train и затем применяются к test. Бинарные one-hot столбцы оставляем как 0/1. Это подготовка для будущих моделей; DummyRegressor в этой работе сами признаки игнорирует.

In [3]:
numeric = ["Relative Compactness", "Surface Area", "Reactor Wall Area",
           "Reactor Roof Area", "Overall Height", "Glazing Area"] + new_features
scale_report = X_train_generated[numeric].agg(["min", "max", "mean", "std"]).T
display(scale_report.round(4))
scaler = StandardScaler()
X_train_scaled = X_train_generated.copy()
X_test_scaled = X_test_generated.copy()
X_train_scaled[numeric] = scaler.fit_transform(X_train_generated[numeric])
X_test_scaled[numeric] = scaler.transform(X_test_generated[numeric])
assert X_train_scaled.columns.equals(X_test_scaled.columns)
assert np.isfinite(X_train_scaled.to_numpy()).all()
assert np.isfinite(X_test_scaled.to_numpy()).all()
print("После масштабирования, статистики train:")
display(X_train_scaled[numeric].agg(["mean", "std"]).T.round(3))

,min,max,mean,std
Relative Compactness,0.6200,0.9800,0.7617,0.1054
Surface Area,514.5000,808.5000,673.8298,88.1612
Reactor Wall Area,245.0000,416.5000,318.8190,43.3360
Reactor Roof Area,110.2500,220.5000,177.5053,44.9602
Overall Height,3.5000,7.0000,5.2215,1.7512
Glazing Area,0.0000,0.4000,0.2354,0.1342
Wall Surface Ratio,0.3571,0.6296,0.4799,0.0815
Roof Surface Ratio,0.1852,0.3214,0.2600,0.0407
Compactness × Glazing,0.0000,0.3920,0.1795,0.1062


После масштабирования, статистики train:


,mean,std
Relative Compactness,0.0,1.001
Surface Area,-0.0,1.001
Reactor Wall Area,-0.0,1.001
Reactor Roof Area,-0.0,1.001
Overall Height,-0.0,1.001
Glazing Area,-0.0,1.001
Wall Surface Ratio,0.0,1.001
Roof Surface Ratio,-0.0,1.001
Compactness × Glazing,-0.0,1.001


**Вывод:** масштабы числовых признаков различаются. После стандартизации средние на train близки к нулю, стандартные отклонения — к единице; небольшие отличия в таблице связаны с определением стандартного отклонения pandas. Test преобразован по статистикам train.

# Раздел 3. Baseline-модель

Используем `DummyRegressor(strategy="mean")`: для каждой из двух нагрузок он предсказывает среднее значение по обучающей части. Это простая отправная точка: более сложная модель должна показать преимущество над ней на тех же данных.

Рассчитываем на test отдельно для каждой цели MAE и RMSE (меньше — лучше), а также R². Для сравнения на этом же тесте предсказание константы, равной среднему именно test, имело бы R² = 0; наша константа получена из train, поэтому R² может оказаться отрицательным. Единицы MAE и RMSE совпадают с единицами целевых значений.

In [4]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_scaled, y_train)
prediction = baseline.predict(X_test_scaled)
rows = []
for i, target in enumerate(y_train.columns):
    truth = y_test[target].to_numpy()
    predicted = prediction[:, i]
    rows.append({"Цель": target,
                 "Среднее train": y_train[target].mean(),
                 "MAE test": mean_absolute_error(truth, predicted),
                 "RMSE test": np.sqrt(mean_squared_error(truth, predicted)),
                 "R² test": r2_score(truth, predicted)})
metrics = pd.DataFrame(rows).set_index("Цель")
display(metrics.round(4))
print("Пример предсказаний:")
display(pd.DataFrame(prediction[:5], columns=y_train.columns, index=y_test.index[:5]).round(3))

,Среднее train,MAE test,RMSE test,R² test
Цель,,,,
Heating Load,22.1550,9.2721,10.2375,-0.0055
Cooling Load,24.4108,8.7419,9.6662,-0.0084


Пример предсказаний:


,Heating Load,Cooling Load
source_row,,
668,22.155,24.411
324,22.155,24.411
624,22.155,24.411
690,22.155,24.411
473,22.155,24.411


## 4. Сохранение и выводы

Сохраняем исходные и масштабированные входы с новыми признаками, цели без изменений, статистики стандартизации и метрики baseline. Признаки созданы одинаково для train и test. В последующих экспериментах выбор модели и её настроек следует делать на обучающих данных; test сохраняется для итогового сравнения. Поскольку здесь test уже использован для baseline, повторный подбор по нему приведёт к утечке информации.

In [5]:
out = Path("processed")
out.mkdir(exist_ok=True)
for name, frame in {"X_train_generated": X_train_generated, "X_test_generated": X_test_generated,
                    "X_train_scaled": X_train_scaled, "X_test_scaled": X_test_scaled,
                    "y_train": y_train, "y_test": y_test}.items():
    frame.to_csv(out / f"{name}.csv", sep=";", index_label="source_row")
pd.DataFrame({"mean": scaler.mean_, "scale": scaler.scale_}, index=numeric).to_csv(
    out / "scaler_parameters.csv", sep=";", index_label="feature")
metrics.to_csv(out / "baseline_metrics.csv", sep=";", index_label="target")
print("Сохранено в", out.resolve())
print("Число признаков:", X_train_scaled.shape[1])
print("Размеры train/test:", len(X_train_scaled), len(X_test_scaled))

Сохранено в /Users/arsenijlogunov/Documents/ChatGPT/Практикум по програмированию/LR4_marvel/processed
Число признаков: 19
Размеры train/test: 614 154


**Итог:** добавлено 3 признака, итоговое число входов — 19. Числовые признаки стандартизованы с параметрами train, one-hot признаки сохранены. Обучена baseline-модель, её качество на test показано для каждой цели.

Для воспроизведения: Python 3.12.5, версии пакетов — в `requirements.txt`; выполняйте ячейки сверху вниз из папки ЛР 4. [Условие ЛР 4](https://wiki.pmifi.ru/ru/disciplines/programming-workshop-bachelors-5sem/labs/generation).